In [2]:
from google.colab import drive
drive.mount('/drive')

import pandas as pd

PROC = '/drive/MyDrive/finshield-ai/data/processed/'
RAW  = '/drive/MyDrive/finshield-ai/data/raw/'

# Vérifie les splits EDA
X_dev  = pd.read_parquet(PROC + 'X_train_raw.parquet')
X_test = pd.read_parquet(PROC + 'X_test_raw.parquet')
y_dev  = pd.read_parquet(PROC + 'y_train.parquet').squeeze()
y_test = pd.read_parquet(PROC + 'y_test.parquet').squeeze()

print(f'X_dev  : {X_dev.shape}')
print(f'X_test : {X_test.shape}')
print(f'Défaut : {y_dev.mean():.2%}')

# Vérifie les tables secondaires
import os
tables = ['bureau','previous_application','installments_payments',
          'credit_card_balance','POS_CASH_balance']
for t in tables:
    path = RAW + t + '.csv'
    exists = os.path.exists(path)
    print(f'  {"✅" if exists else "❌"} {t}')

Mounted at /drive
X_dev  : (246008, 120)
X_test : (61503, 120)
Défaut : 8.07%
  ✅ bureau
  ❌ previous_application
  ❌ installments_payments
  ❌ credit_card_balance
  ❌ POS_CASH_balance


In [3]:
bureau = pd.read_csv(RAW + 'bureau.csv')
print(f'bureau : {bureau.shape}')

bureau : (1716428, 17)


In [4]:
import numpy as np
from sklearn.model_selection import train_test_split

# ── FEATURE ENGINEERING TABLE PRINCIPALE ──────────────
def fe_main(df):
    df = df.copy()

    # Flags MNAR
    for col in ['EXT_SOURCE_1','EXT_SOURCE_3','OWN_CAR_AGE']:
        df[f'{col}_MISSING'] = df[col].isnull().astype(int)

    # Age et emploi
    df['AGE_YEARS']      = -df['DAYS_BIRTH'] / 365
    df['DAYS_EMPLOYED']  = df['DAYS_EMPLOYED'].replace(365243, np.nan)
    df['YEARS_EMPLOYED'] = -df['DAYS_EMPLOYED'] / 365
    df['SENIORITY_RATIO']= df['YEARS_EMPLOYED'] / (df['AGE_YEARS'] + 1)

    # Ratios financiers
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY']  / (df['AMT_INCOME_TOTAL'] + 1)
    df['CREDIT_INCOME_RATIO']  = df['AMT_CREDIT']   / (df['AMT_INCOME_TOTAL'] + 1)
    df['CREDIT_GOODS_RATIO']   = df['AMT_CREDIT']   / (df['AMT_GOODS_PRICE']  + 1)
    df['ANNUITY_CREDIT_RATIO'] = df['AMT_ANNUITY']  / (df['AMT_CREDIT']       + 1)
    df['INCOME_PER_PERSON']    = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'] + 1)

    # EXT_SOURCE
    ext = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']
    df['EXT_SOURCE_MEAN']    = df[ext].mean(axis=1)
    df['EXT_SOURCE_MIN']     = df[ext].min(axis=1)
    df['EXT_SOURCE_STD']     = df[ext].std(axis=1)
    df['EXT_SOURCE_PROD']    = df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']
    df['EXT_MEAN_X_AGE']     = df['EXT_SOURCE_MEAN'] * df['AGE_YEARS']

    # Comptages
    doc_cols = [c for c in df.columns if 'FLAG_DOCUMENT' in c]
    df['DOCUMENTS_COUNT'] = df[doc_cols].sum(axis=1)

    return df

X_dev  = fe_main(X_dev)
X_test = fe_main(X_test)
print(f'Features après FE main : {X_dev.shape[1]}')

Features après FE main : 137


In [5]:
# ── FEATURE ENGINEERING BUREAU ────────────────────────
def fe_bureau(bureau):
    agg = bureau.groupby('SK_ID_CURR').agg(
        BUREAU_LOAN_COUNT       = ('SK_ID_BUREAU', 'count'),
        BUREAU_AMT_CREDIT_SUM   = ('AMT_CREDIT_SUM', 'sum'),
        BUREAU_AMT_CREDIT_MEAN  = ('AMT_CREDIT_SUM', 'mean'),
        BUREAU_AMT_DEBT_SUM     = ('AMT_CREDIT_SUM_DEBT', 'sum'),
        BUREAU_AMT_DEBT_MEAN    = ('AMT_CREDIT_SUM_DEBT', 'mean'),
        BUREAU_OVERDUE_COUNT    = ('CREDIT_DAY_OVERDUE', lambda x: (x>0).sum()),
        BUREAU_MAX_OVERDUE      = ('CREDIT_DAY_OVERDUE', 'max'),
        BUREAU_ACTIVE_COUNT     = ('CREDIT_ACTIVE', lambda x: (x=='Active').sum()),
        BUREAU_CLOSED_COUNT     = ('CREDIT_ACTIVE', lambda x: (x=='Closed').sum()),
        BUREAU_CREDIT_TYPES     = ('CREDIT_TYPE', 'nunique'),
        BUREAU_DAYS_CREDIT_MEAN = ('DAYS_CREDIT', 'mean'),
        BUREAU_DAYS_CREDIT_MIN  = ('DAYS_CREDIT', 'min'),
    ).reset_index()

    agg['BUREAU_DEBT_CREDIT_RATIO'] = agg['BUREAU_AMT_DEBT_SUM'] / (agg['BUREAU_AMT_CREDIT_SUM'] + 1)
    agg['BUREAU_ACTIVE_RATIO']      = agg['BUREAU_ACTIVE_COUNT'] / (agg['BUREAU_LOAN_COUNT'] + 1)
    agg['BUREAU_OVERDUE_RATIO']     = agg['BUREAU_OVERDUE_COUNT'] / (agg['BUREAU_LOAN_COUNT'] + 1)

    return agg

bureau_agg = fe_bureau(bureau)

X_dev  = X_dev.merge(bureau_agg, on='SK_ID_CURR', how='left') if 'SK_ID_CURR' in X_dev.columns else X_dev
X_test = X_test.merge(bureau_agg, on='SK_ID_CURR', how='left') if 'SK_ID_CURR' in X_test.columns else X_test

print(f'Features après bureau : {X_dev.shape[1]}')

Features après bureau : 137


In [6]:
print('SK_ID_CURR' in X_dev.columns)
print(X_dev.columns[:5].tolist())

False
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN']


In [7]:
# Recharge la table principale complète
app = pd.read_csv(RAW + 'application_train.csv')

# Feature engineering
app = fe_main(app)

# Merge bureau
app = app.merge(bureau_agg, on='SK_ID_CURR', how='left')

print(f'Shape après merge : {app.shape}')

# Re-split proprement
from sklearn.model_selection import train_test_split

y = app['TARGET']
X = app.drop(columns=['TARGET', 'SK_ID_CURR'])

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f'X_dev  : {X_dev.shape}')
print(f'X_test : {X_test.shape}')
print(f'Défaut : {y_dev.mean():.2%}')

Shape après merge : (307511, 154)
X_dev  : (246008, 152)
X_test : (61503, 152)
Défaut : 8.07%


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Supprime colonnes >50% NaN
missing_pct  = X_dev.isnull().mean()
protected    = [c for c in X_dev.columns if '_MISSING' in c or
                c in ['EXT_SOURCE_1','EXT_SOURCE_3','OWN_CAR_AGE']]
cols_to_drop = [c for c in missing_pct[missing_pct > 0.5].index
                if c not in protected]
X_dev  = X_dev.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

# Colonnes constantes
constant_cols = [c for c in X_dev.columns if X_dev[c].nunique() <= 1]
X_dev  = X_dev.drop(columns=constant_cols)
X_test = X_test.drop(columns=constant_cols)

cat_cols = X_dev.select_dtypes(include='object').columns.tolist()
num_cols = X_dev.select_dtypes(include=np.number).columns.tolist()

neg = (y_dev==0).sum()
pos = (y_dev==1).sum()
spw = neg / pos

print(f'Features finales   : {X_dev.shape[1]}')
print(f'Numériques         : {len(num_cols)}')
print(f'Catégorielles      : {len(cat_cols)}')
print(f'scale_pos_weight   : {spw:.2f}')

Features finales   : 113
Numériques         : 100
Catégorielles      : 13
scale_pos_weight   : 11.39


In [9]:
# ── KNN FEATURE ───────────────────────────────────────
from sklearn.neighbors import KNeighborsClassifier

knn_cols = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3','CREDIT_INCOME_RATIO']
X_knn_dev  = X_dev[knn_cols].fillna(0)
X_knn_test = X_test[knn_cols].fillna(0)

knn = KNeighborsClassifier(n_neighbors=500, n_jobs=-1, metric='euclidean')
knn.fit(X_knn_dev, y_dev)

X_dev['NEIGHBORS_TARGET_MEAN_500']  = knn.predict_proba(X_knn_dev)[:, 1]
X_test['NEIGHBORS_TARGET_MEAN_500'] = knn.predict_proba(X_knn_test)[:, 1]

print('KNN feature ok')

# ── K-MEANS EXT_SOURCE ────────────────────────────────
from sklearn.cluster import KMeans

ext_cols = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']
X_ext = X_dev[ext_cols].fillna(X_dev[ext_cols].median())

for k in [3, 5, 9]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    X_dev[f'EXT_CLUSTER_{k}']  = km.fit_predict(X_ext).astype(str)
    X_test[f'EXT_CLUSTER_{k}'] = km.predict(
        X_test[ext_cols].fillna(X_dev[ext_cols].median())
    ).astype(str)

print('K-Means ok')

# ── GROUPED MEANS ─────────────────────────────────────
# Calculé UNIQUEMENT sur X_dev — zéro leakage
train_with_target = X_dev.copy()
train_with_target['TARGET'] = y_dev.values

for cat in ['NAME_INCOME_TYPE','NAME_EDUCATION_TYPE','OCCUPATION_TYPE']:
    if cat in X_dev.columns:
        means = train_with_target.groupby(cat)['TARGET'].mean()
        X_dev[f'{cat}_TARGET_MEAN']  = X_dev[cat].map(means)
        X_test[f'{cat}_TARGET_MEAN'] = X_test[cat].map(means)

print('Grouped means ok')

# Update colonnes
cat_cols = X_dev.select_dtypes(include='object').columns.tolist()
num_cols = X_dev.select_dtypes(include=np.number).columns.tolist()
print(f'Features finales : {X_dev.shape[1]}')

KNN feature ok
K-Means ok
Grouped means ok
Features finales : 120


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

cat_cols = X_dev.select_dtypes(include='object').columns.tolist()
num_cols = X_dev.select_dtypes(include=np.number).columns.tolist()

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
], remainder='drop')

print('Preprocesseur ok')
print(f'Numériques : {len(num_cols)}')
print(f'Catégorielles : {len(cat_cols)}')

Preprocesseur ok
Numériques : 104
Catégorielles : 16


In [12]:
import lightgbm as lgb
import xgboost as xgb
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe_lgb = Pipeline([
    ('pre', preprocessor),
    ('model', lgb.LGBMClassifier(
        n_estimators=1000, max_depth=6, learning_rate=0.05,
        num_leaves=63, subsample=0.8, colsample_bytree=0.7,
        is_unbalance=True, random_state=42, n_jobs=-1, verbose=-1,
        device='gpu'
    ))
])

pipe_xgb = Pipeline([
    ('pre', preprocessor),
    ('model', xgb.XGBClassifier(
        n_estimators=1000, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7,
        scale_pos_weight=spw,
        eval_metric='auc', random_state=42,
        tree_method='hist', device='cuda'
    ))
])

print('Modèles configurés — lancement CV...')

# CV LightGBM
lgb_scores = cross_val_score(pipe_lgb, X_dev, y_dev, cv=CV, scoring='roc_auc', n_jobs=-1)
print(f'LGB AUC : {lgb_scores.mean():.4f} ± {lgb_scores.std():.4f}')

# CV XGBoost
xgb_scores = cross_val_score(pipe_xgb, X_dev, y_dev, cv=CV, scoring='roc_auc', n_jobs=-1)
print(f'XGB AUC : {xgb_scores.mean():.4f} ± {xgb_scores.std():.4f}')

Modèles configurés — lancement CV...
LGB AUC : 0.7617 ± 0.0026


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


XGB AUC : 0.7689 ± 0.0020


In [13]:
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.preprocessing import OrdinalEncoder
import warnings
warnings.filterwarnings('ignore')

# Transforme X_dev pour EasyEnsemble
pre_fit   = preprocessor.__class__(
    transformers=preprocessor.transformers, remainder='drop'
)
X_dev_t  = pre_fit.fit_transform(X_dev)
X_test_t = pre_fit.transform(X_test)

# EasyEnsemble
ee = EasyEnsembleClassifier(n_estimators=20, random_state=42, n_jobs=-1)
ee_scores = cross_val_score(ee, X_dev_t, y_dev, cv=CV, scoring='roc_auc', n_jobs=-1)
print(f'EasyEnsemble AUC : {ee_scores.mean():.4f} ± {ee_scores.std():.4f}')

EasyEnsemble AUC : 0.7517 ± 0.0018


In [14]:
# Entraînement final sur X_dev complet
print('Training final LightGBM...')
pipe_lgb.fit(X_dev, y_dev)

print('Training final XGBoost...')
pipe_xgb.fit(X_dev, y_dev)

print('Training final EasyEnsemble...')
ee.fit(X_dev_t, y_dev)

print('Tous les modèles entraînés')

Training final LightGBM...
Training final XGBoost...
Training final EasyEnsemble...
Tous les modèles entraînés


In [15]:
from sklearn.metrics import (
    roc_auc_score, f1_score, average_precision_score,
    classification_report, precision_score, recall_score
)
import numpy as np

# Prédictions
y_proba_lgb = pipe_lgb.predict_proba(X_test)[:, 1]
y_proba_xgb = pipe_xgb.predict_proba(X_test)[:, 1]
y_proba_ee  = ee.predict_proba(X_test_t)[:, 1]

def evaluate(y_true, y_proba, name):
    auc  = roc_auc_score(y_true, y_proba)
    ap   = average_precision_score(y_true, y_proba)
    gini = 2*auc - 1

    # Seuil optimal F1
    ths  = np.arange(0.01, 0.90, 0.005)
    f1s  = [f1_score(y_true, (y_proba>=t).astype(int), pos_label=1, zero_division=0) for t in ths]
    best_t = ths[np.argmax(f1s)]
    best_f1 = max(f1s)

    y_pred = (y_proba >= best_t).astype(int)

    print(f'━━━ {name} ━━━')
    print(f'AUC-ROC  : {auc:.4f}')
    print(f'Gini     : {gini:.4f}')
    print(f'Avg Prec : {ap:.4f}')
    print(f'F1 (cl.1): {best_f1:.4f}  seuil={best_t:.3f}')
    print(f'Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}')
    print(f'Recall   : {recall_score(y_true, y_pred, zero_division=0):.4f}')
    print()

evaluate(y_test, y_proba_lgb, 'LightGBM')
evaluate(y_test, y_proba_xgb, 'XGBoost')
evaluate(y_test, y_proba_ee,  'EasyEnsemble')

━━━ LightGBM ━━━
AUC-ROC  : 0.7658
Gini     : 0.5315
Avg Prec : 0.2595
F1 (cl.1): 0.3190  seuil=0.610
Precision: 0.2477
Recall   : 0.4479

━━━ XGBoost ━━━
AUC-ROC  : 0.7716
Gini     : 0.5432
Avg Prec : 0.2685
F1 (cl.1): 0.3229  seuil=0.640
Precision: 0.2542
Recall   : 0.4425

━━━ EasyEnsemble ━━━
AUC-ROC  : 0.7522
Gini     : 0.5044
Avg Prec : 0.2420
F1 (cl.1): 0.2990  seuil=0.580
Precision: 0.2408
Recall   : 0.3944



In [16]:
cc = pd.read_csv(RAW + 'credit_card_balance.csv')
print(f'credit_card_balance : {cc.shape}')

credit_card_balance : (3840312, 23)


In [17]:
# ── AGRÉGATION CREDIT CARD BALANCE ───────────────────
def fe_credit_card(cc):
    cc['UTILIZATION'] = cc['AMT_BALANCE'] / (cc['AMT_CREDIT_LIMIT_ACTUAL'] + 1)

    agg = cc.groupby('SK_ID_CURR').agg(
        CC_COUNT            = ('SK_ID_PREV', 'nunique'),
        CC_AMT_BALANCE_MEAN = ('AMT_BALANCE', 'mean'),
        CC_AMT_BALANCE_MAX  = ('AMT_BALANCE', 'max'),
        CC_LIMIT_MEAN       = ('AMT_CREDIT_LIMIT_ACTUAL', 'mean'),
        CC_UTILIZATION_MEAN = ('UTILIZATION', 'mean'),
        CC_UTILIZATION_MAX  = ('UTILIZATION', 'max'),
        CC_DPD_MEAN         = ('SK_DPD', 'mean'),
        CC_DPD_MAX          = ('SK_DPD', 'max'),
        CC_DPD_COUNT        = ('SK_DPD', lambda x: (x>0).sum()),
        CC_DRAWINGS_MEAN    = ('AMT_DRAWINGS_CURRENT', 'mean'),
    ).reset_index()

    agg['CC_HIGH_UTILIZATION'] = (agg['CC_UTILIZATION_MAX'] > 0.8).astype(int)
    agg['CC_DPD_RATIO']        = agg['CC_DPD_COUNT'] / (agg['CC_COUNT'] + 1)

    return agg

cc_agg = fe_credit_card(cc)
print(f'CC agrégé : {cc_agg.shape}')

# Repart de zéro avec les 3 tables
app = pd.read_csv(RAW + 'application_train.csv')
app = fe_main(app)
app = app.merge(bureau_agg, on='SK_ID_CURR', how='left')
app = app.merge(cc_agg,     on='SK_ID_CURR', how='left')

y = app['TARGET']
X = app.drop(columns=['TARGET', 'SK_ID_CURR'])

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f'Shape final : {X_dev.shape}')

CC agrégé : (103558, 13)
Shape final : (246008, 164)


In [18]:
# ── NETTOYAGE ─────────────────────────────────────────
missing_pct  = X_dev.isnull().mean()
protected    = [c for c in X_dev.columns if '_MISSING' in c or
                c in ['EXT_SOURCE_1','EXT_SOURCE_3','OWN_CAR_AGE']]
cols_to_drop = [c for c in missing_pct[missing_pct > 0.5].index
                if c not in protected]
X_dev  = X_dev.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

constant_cols = [c for c in X_dev.columns if X_dev[c].nunique() <= 1]
X_dev  = X_dev.drop(columns=constant_cols)
X_test = X_test.drop(columns=constant_cols)

# ── KNN FEATURE ───────────────────────────────────────
from sklearn.neighbors import KNeighborsClassifier
knn_cols   = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3','CREDIT_INCOME_RATIO']
knn        = KNeighborsClassifier(n_neighbors=500, n_jobs=-1)
knn.fit(X_dev[knn_cols].fillna(0), y_dev)
X_dev['NEIGHBORS_TARGET_MEAN_500']  = knn.predict_proba(X_dev[knn_cols].fillna(0))[:, 1]
X_test['NEIGHBORS_TARGET_MEAN_500'] = knn.predict_proba(X_test[knn_cols].fillna(0))[:, 1]

# ── K-MEANS EXT_SOURCE ────────────────────────────────
from sklearn.cluster import KMeans
ext_cols = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']
X_ext    = X_dev[ext_cols].fillna(X_dev[ext_cols].median())
for k in [3, 5, 9]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    X_dev[f'EXT_CLUSTER_{k}']  = km.fit_predict(X_ext).astype(str)
    X_test[f'EXT_CLUSTER_{k}'] = km.predict(X_test[ext_cols].fillna(X_dev[ext_cols].median())).astype(str)

# ── GROUPED MEANS ─────────────────────────────────────
tmp = X_dev.copy()
tmp['TARGET'] = y_dev.values
for cat in ['NAME_INCOME_TYPE','NAME_EDUCATION_TYPE','OCCUPATION_TYPE']:
    if cat in X_dev.columns:
        means = tmp.groupby(cat)['TARGET'].mean()
        X_dev[f'{cat}_TARGET_MEAN']  = X_dev[cat].map(means)
        X_test[f'{cat}_TARGET_MEAN'] = X_test[cat].map(means)

# ── PIPELINE ──────────────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

cat_cols = X_dev.select_dtypes(include='object').columns.tolist()
num_cols = X_dev.select_dtypes(include=np.number).columns.tolist()
neg = (y_dev==0).sum(); pos = (y_dev==1).sum(); spw = neg/pos

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  RobustScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('enc', OrdinalEncoder(handle_unknown='use_encoded_value',
                                             unknown_value=-1))]), cat_cols)
], remainder='drop')

pipe_xgb = Pipeline([
    ('pre', preprocessor),
    ('model', xgb.XGBClassifier(
        n_estimators=1000, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7,
        scale_pos_weight=spw, eval_metric='auc',
        random_state=42, n_jobs=-1, tree_method='hist', device='cuda'
    ))
])

pipe_lgb = Pipeline([
    ('pre', preprocessor),
    ('model', lgb.LGBMClassifier(
        n_estimators=1000, max_depth=6, learning_rate=0.05,
        num_leaves=63, subsample=0.8, colsample_bytree=0.7,
        is_unbalance=True, random_state=42, n_jobs=-1, verbose=-1,
        device='gpu'
    ))
])

print(f'Features finales : {X_dev.shape[1]}')
print(f'scale_pos_weight : {spw:.2f}')

Features finales : 120
scale_pos_weight : 11.39


In [19]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('CV LightGBM...')
lgb_scores = cross_val_score(pipe_lgb, X_dev, y_dev, cv=CV, scoring='roc_auc', n_jobs=-1)
print(f'LGB AUC : {lgb_scores.mean():.4f} ± {lgb_scores.std():.4f}')

print('CV XGBoost...')
xgb_scores = cross_val_score(pipe_xgb, X_dev, y_dev, cv=CV, scoring='roc_auc', n_jobs=-1)
print(f'XGB AUC : {xgb_scores.mean():.4f} ± {xgb_scores.std():.4f}')

print('Training final...')
pipe_lgb.fit(X_dev, y_dev)
pipe_xgb.fit(X_dev, y_dev)
print('✅ Done')

CV LightGBM...
LGB AUC : 0.7617 ± 0.0026
CV XGBoost...
XGB AUC : 0.7689 ± 0.0020
Training final...
✅ Done
